# DDT vs Boosting: Airlines Departure Delay Benchmark Profiles

Compares **DDT** against LightGBM, XGBoost, and CatBoost on [Airlines_DepDelay_1M](https://www.openml.org/d/42452).

**Measurement principles:**
- Temporal hold-out split (train = first 80%, test = last 20%) to avoid route-level leakage.
- Subsample drawn from training set only, after the temporal split.
- CVaR uses a fixed empirical threshold (true P95 of the test set).
- Model size via `pickle.dumps` (note: LGBM pickle includes Python wrapper overhead).


In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, TargetEncoder
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from ddt import DDTRegressor
import time
import pickle
from IPython.display import display, HTML

# --- Metrics ---
def calc_cce(y_true, q_lower, q_upper, target_coverage):
    empirical_coverage = np.mean((y_true >= q_lower) & (y_true <= q_upper))
    return abs(target_coverage - empirical_coverage)

def calc_apiw(q_lower, q_upper):
    return np.mean(q_upper - q_lower)

def calc_cvar(y_true, q_pred):
    """CVaR computed with the 95% of the predicted."""
    breaches = y_true[y_true > q_pred]
    return float(np.mean(breaches)) if len(breaches) > 0 else 0.0

def calc_hei(infer_time_sec, n_samples, model_obj):
    latency_us = (infer_time_sec / n_samples) * 1e6
    size_kb = len(pickle.dumps(model_obj)) / 1024
    return latency_us * size_kb


In [ ]:
BENCHMARK_PROFILES = {
    "1_vs_1_theory": {
        "name": "1-vs-1 Representational Capacity (The Theoretical Duel)",
        "description": (
            "Intrinsic capacity of a single tree without ensembling. "
            "All models: 1 tree per quantile."
        ),
        "target_quantiles": [0.05, 0.50, 0.95],
        "ddt_params": {
            "max_depth": 5, "min_samples_leaf": 200,
            "n_target_bins": 64, "n_feature_bins": 64,
        },
        # DDT-Cal: DDT-Hybrid + marginal conformal calibration.
        # Adjust calibration_fraction / calibration_mode to experiment.
        "ddt_cal_params": {
            "calibration_fraction": 0.15,   # fraction held out for calibration
            "calibration_mode": "hybrid",  # 'marginal' | 'hybrid' (leaf-level)
            "core_fraction": 0.5,            # hybrid-bin core fraction
        },
        "competitors_params": {
            "n_estimators": 1, "max_depth": 5, "num_leaves": 32,
        }
    },

    "edge_microkernel": {
        "name": "Edge AI & Micro-Kernel (The Silicon Test)",
        "description": (
            "Constrained hardware (IoT/HFT). All models: 1 tree per quantile (true 1-vs-1). "
            "DDT produces all quantiles in one pass; GBMs use separate per-quantile trees."
        ),
        "target_quantiles": [0.05, 0.50, 0.95],
        "ddt_params": {
            "max_depth": 5, "min_samples_leaf": 100,
            "n_target_bins": 64, "n_feature_bins": 32,
        },
        # DDT-Cal: DDT-Hybrid + marginal conformal calibration.
        # Adjust calibration_fraction / calibration_mode to experiment.
        "ddt_cal_params": {
            "calibration_fraction": 0.15,   # fraction held out for calibration
            "calibration_mode": "marginal",  # 'marginal' | 'hybrid' (leaf-level)
            "core_fraction": 0.5,            # hybrid-bin core fraction
        },
        "competitors_params": {
            # Fixed: was 3 (unfair vs DDT single tree). Now 1 per quantile.
            "n_estimators": 1, "max_depth": 5, "num_leaves": 32,
        }
    },

    "production_ensemble": {
        "name": "Production Battle (Single-Tree DDT vs. 100-Tree Boosting)",
        "description": (
            "Single DDT vs 100-tree GBM ensembles. "
            "GBMs train 3x100-tree models (one per quantile): 300 trees total. "
            "Model size reflects full multi-quantile deployment cost."
        ),
        "target_quantiles": [0.05, 0.50, 0.95],
        "ddt_params": {
            "max_depth": 8, "min_samples_leaf": 200,
            "n_target_bins": 64, "n_feature_bins": 64,
        },
        # DDT-Cal: DDT-Hybrid + marginal conformal calibration.
        # Adjust calibration_fraction / calibration_mode to experiment.
        "ddt_cal_params": {
            "calibration_fraction": 0.15,   # fraction held out for calibration
            "calibration_mode": "marginal",  # 'marginal' | 'hybrid' (leaf-level)
            "core_fraction": 0.5,            # hybrid-bin core fraction
        },
        "competitors_params": {
            "n_estimators": 100, "max_depth": 8, "learning_rate": 0.05,
        }
    },

    "bimodal_distribution": {
        "name": "Bimodal Distribution Stress Test (DDT Core Strength)",
        "description": (
            "Airlines delays are bimodal: most flights near on-time, "
            "but a subset have long tarmac/mechanical delays (second mode at +60-180 min). "
            "n_target_bins=80 gives DDT full distributional expressiveness. "
            "min_samples_leaf=500 ensures smooth, stable ECDFs. "
            "GBMs receive 50 trees for competitive footing."
        ),
        "target_quantiles": [0.05, 0.50, 0.95],
        "ddt_params": {
            "max_depth": 7, "min_samples_leaf": 500,
            "n_target_bins": 80, "n_feature_bins": 64,
        },
        # DDT-Cal: DDT-Hybrid + marginal conformal calibration.
        # Adjust calibration_fraction / calibration_mode to experiment.
        "ddt_cal_params": {
            "calibration_fraction": 0.15,   # fraction held out for calibration
            "calibration_mode": "marginal",  # 'marginal' | 'hybrid' (leaf-level)
            "core_fraction": 0.5,            # hybrid-bin core fraction
        },
        "competitors_params": {
            "n_estimators": 50, "max_depth": 7, "learning_rate": 0.05,
        }
    },

}

In [13]:
# --- DATA LOADING & PREPROCESSING ---
# Temporal split: train = first 80%, test = last 20%.
# Subsample drawn from training set ONLY, after split.
MAX_TRAIN_SAMPLES = 100000

from sklearn.preprocessing import MinMaxScaler
air = fetch_openml(name="Airlines_DepDelay_1M", version=1, as_frame=True, parser="auto")
X_all = air.data.copy()
y_all = air.target.values.astype(float)

valid = ~np.isnan(y_all)
X_all = X_all[valid].reset_index(drop=True)
y_all = y_all[valid]

n_train_full = int(0.8 * len(X_all))
X_train_full = X_all.iloc[:n_train_full].reset_index(drop=True)
y_train_full = y_all[:n_train_full]
X_test_raw   = X_all.iloc[n_train_full:].reset_index(drop=True)
y_test       = y_all[n_train_full:]

if MAX_TRAIN_SAMPLES is not None and len(X_train_full) > MAX_TRAIN_SAMPLES:
    rng = np.random.default_rng(42)
    idx = rng.choice(len(X_train_full), MAX_TRAIN_SAMPLES, replace=False)
    X_train_raw = X_train_full.iloc[idx].reset_index(drop=True)
    y_train     = y_train_full[idx]
else:
    X_train_raw = X_train_full
    y_train     = y_train_full

print(f"Train: {len(X_train_raw):,} rows | Test: {len(X_test_raw):,} rows")

# Fixed CVaR threshold: true P95 of the test set (computed before any model runs)
EMPIRICAL_P95_TEST = float(np.percentile(y_test, 95))
print(f"Fixed CVaR threshold (true test P95): {EMPIRICAL_P95_TEST:.1f} min")

categorical_cols = X_train_raw.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_cols   = X_train_raw.select_dtypes(include=[np.number]).columns.tolist()

def preprocess_pipeline(X_train_df, X_test_df, y_train_arr,
                         target_strategy="winsorized", feature_scaling="none"):
    X_tr = X_train_df.copy()
    X_te = X_test_df.copy()
    y_tr = y_train_arr.copy()

    if len(categorical_cols) > 0:
        enc = TargetEncoder(smooth="auto", cv=5, random_state=42, target_type="continuous")
        X_tr_cat = enc.fit_transform(X_tr[categorical_cols], y_train_arr)
        X_te_cat = enc.transform(X_te[categorical_cols])
        X_tr_num = X_tr[numerical_cols].fillna(0).values
        X_te_num = X_te[numerical_cols].fillna(0).values
        X_train_out = np.hstack([X_tr_num, X_tr_cat])
        X_test_out  = np.hstack([X_te_num, X_te_cat])
    else:
        X_train_out = X_tr[numerical_cols].fillna(0).values
        X_test_out  = X_te[numerical_cols].fillna(0).values

    if feature_scaling == "standard":
        sc = StandardScaler()
        X_train_out = sc.fit_transform(X_train_out)
        X_test_out  = sc.transform(X_test_out)
    elif feature_scaling == "minmax":
        sc = MinMaxScaler()
        X_train_out = sc.fit_transform(X_train_out)
        X_test_out  = sc.transform(X_test_out)
    # "none": tree models are invariant to monotone feature transforms

    y_scaler = None
    if target_strategy == "winsorized":
        lo, hi = np.percentile(y_tr, 1), np.percentile(y_tr, 99)
        y_tr = np.clip(y_tr, lo, hi)
    if target_strategy in ["standard", "winsorized"]:
        y_scaler = StandardScaler()
        y_tr = y_scaler.fit_transform(y_tr.reshape(-1, 1)).flatten()

    return X_train_out, X_test_out, y_tr, y_scaler


Train: 100,000 rows | Test: 200,000 rows
Fixed CVaR threshold (true test P95): 63.0 min


In [14]:
def run_benchmark(profile_name, profile, X_train_scaled, X_test_scaled, y_train, y_test, preproc_name, y_scaler=None):
    sep = "=" * 80
    print(f"\n{sep}\nProfile: {profile['name']} | Preprocessing: {preproc_name}\n{sep}")
    n_test = len(y_test)
    ddt_p  = profile["ddt_params"].copy()
    comp_p = profile["competitors_params"].copy()

    # --- DDT ---
    ddt = DDTRegressor(**ddt_p)
    t0 = time.time(); ddt.fit(X_train_scaled, y_train); ddt_fit_t = time.time() - t0
    t0 = time.time()
    ddt_preds = ddt.predict_quantiles(X_test_scaled, profile["target_quantiles"])
    ddt_inf_t = time.time() - t0
    ddt_q05 = ddt_preds[profile["target_quantiles"][0]]
    ddt_q95 = ddt_preds[profile["target_quantiles"][2]]

    # --- DDT-Log ---
    ddt_log_p = ddt_p.copy()
    ddt_log_p["bin_strategy"] = "log_width"
    ddt_log_p["winsorize_tails"] = 0.001
    
    offset = abs(np.min(y_train)) + 1.0 if np.min(y_train) <= 0 else 0.0
    y_train_log = y_train + offset
    
    ddt_log = DDTRegressor(**ddt_log_p)
    t0 = time.time(); ddt_log.fit(X_train_scaled, y_train_log); ddt_log_fit_t = time.time() - t0
    t0 = time.time()
    ddt_log_preds = ddt_log.predict_quantiles(X_test_scaled, profile["target_quantiles"])
    ddt_log_inf_t = time.time() - t0
    ddt_log_q05 = ddt_log_preds[profile["target_quantiles"][0]] - offset
    ddt_log_q95 = ddt_log_preds[profile["target_quantiles"][2]] - offset

    # --- DDT-Hybrid ---
    ddt_hyb_p = ddt_p.copy()
    ddt_hyb_p["bin_strategy"] = "hybrid"
    ddt_hyb_p["core_fraction"] = 0.5
    
    ddt_hyb = DDTRegressor(**ddt_hyb_p)
    t0 = time.time(); ddt_hyb.fit(X_train_scaled, y_train); ddt_hyb_fit_t = time.time() - t0
    t0 = time.time()
    ddt_hyb_preds = ddt_hyb.predict_quantiles(X_test_scaled, profile["target_quantiles"])
    ddt_hyb_inf_t = time.time() - t0
    ddt_hyb_q05 = ddt_hyb_preds[profile["target_quantiles"][0]]
    ddt_hyb_q95 = ddt_hyb_preds[profile["target_quantiles"][2]]

    # --- DDT-Cal (DDT-Hybrid + marginal conformal calibration) ---
    # Calibration parameters come from profile["ddt_cal_params"].
    # DDTRegressor.fit() internally holds out `calibration_fraction` of the
    # training data; predict_quantiles() then applies per-quantile residual shifts.
    cal_overrides = profile.get("ddt_cal_params", {})
    ddt_cal_p = ddt_p.copy()
    ddt_cal_p["bin_strategy"]         = "hybrid"
    ddt_cal_p["core_fraction"]        = cal_overrides.get("core_fraction", 0.5)
    ddt_cal_p["calibration_fraction"] = cal_overrides.get("calibration_fraction", 0.15)
    ddt_cal_p["calibration_mode"]     = cal_overrides.get("calibration_mode", "hybrid")

    ddt_cal = DDTRegressor(**ddt_cal_p)
    t0 = time.time(); ddt_cal.fit(X_train_scaled, y_train); ddt_cal_fit_t = time.time() - t0
    t0 = time.time()
    ddt_cal_preds = ddt_cal.predict_quantiles(X_test_scaled, profile["target_quantiles"])
    ddt_cal_inf_t = time.time() - t0
    ddt_cal_q05 = ddt_cal_preds[profile["target_quantiles"][0]]
    ddt_cal_q95 = ddt_cal_preds[profile["target_quantiles"][2]]

    # --- LightGBM ---
    lgbms = []
    t0 = time.time()
    for q in profile["target_quantiles"]:
        m = LGBMRegressor(objective="quantile", alpha=q, random_state=42, verbose=-1, **comp_p)
        m.fit(X_train_scaled, y_train); lgbms.append(m)
    lgbm_fit_t = time.time() - t0
    t0 = time.time()
    lgbm_q05 = lgbms[0].predict(X_test_scaled)
    lgbm_q95 = lgbms[2].predict(X_test_scaled)
    lgbm_inf_t = time.time() - t0

    # --- XGBoost ---
    xgbs = []
    t0 = time.time()
    for q in profile["target_quantiles"]:
        xp = comp_p.copy()
        if "num_leaves" in xp:
            xp["max_leaves"] = xp.pop("num_leaves")
            xp["grow_policy"] = "lossguide"
        m = XGBRegressor(objective="reg:quantileerror", quantile_alpha=q, random_state=42, **xp)
        m.fit(X_train_scaled, y_train); xgbs.append(m)
    xgb_fit_t = time.time() - t0
    t0 = time.time()
    xgb_q05 = xgbs[0].predict(X_test_scaled)
    xgb_q95 = xgbs[2].predict(X_test_scaled)
    xgb_inf_t = time.time() - t0

    # --- CatBoost ---
    qs_str = ",".join(str(q) for q in profile["target_quantiles"])
    cp = comp_p.copy()
    cp["max_depth"] = min(cp["max_depth"], 16)
    if "n_estimators" in cp: cp["iterations"] = cp.pop("n_estimators")
    if "num_leaves"   in cp: cp.pop("num_leaves")
    cat = CatBoostRegressor(loss_function=f"MultiQuantile:alpha={qs_str}", random_seed=42, verbose=False, **cp)
    t0 = time.time(); cat.fit(X_train_scaled, y_train); cat_fit_t = time.time() - t0
    t0 = time.time(); cp_out = cat.predict(X_test_scaled); cat_inf_t = time.time() - t0
    cat_q05 = cp_out[:, 0]; cat_q95 = cp_out[:, 2]

    # --- Inverse transform ---
    if y_scaler is not None:
        def inv(a): return y_scaler.inverse_transform(a.reshape(-1, 1)).flatten()
        ddt_q05,  ddt_q95  = inv(ddt_q05),  inv(ddt_q95)
        ddt_log_q05, ddt_log_q95 = inv(ddt_log_q05), inv(ddt_log_q95)
        ddt_hyb_q05, ddt_hyb_q95 = inv(ddt_hyb_q05), inv(ddt_hyb_q95)
        ddt_cal_q05, ddt_cal_q95 = inv(ddt_cal_q05), inv(ddt_cal_q95)
        lgbm_q05, lgbm_q95 = inv(lgbm_q05), inv(lgbm_q95)
        xgb_q05,  xgb_q95  = inv(xgb_q05),  inv(xgb_q95)
        cat_q05,  cat_q95  = inv(cat_q05),  inv(cat_q95)

    # --- Metrics ---
    target_cov  = profile["target_quantiles"][2] - profile["target_quantiles"][0]
    tail_pct    = int(profile["target_quantiles"][2] * 100)
    # Fixed empirical threshold — NOT the per-sample model predicted P95
    cvar_thresh = EMPIRICAL_P95_TEST

    def fmt_row(q05, q95, inf_t, obj, fit_t):
        return [
            f"{calc_cce(y_test, q05, q95, target_cov)*100:.2f}%",
            f"{calc_apiw(q05, q95):,.2f}",
            f"{calc_cvar(y_test, q95):,.2f}",
            f"{fit_t:.2f}",
            f"{(inf_t / n_test) * 1e6:.2f}",
            f"{len(pickle.dumps(obj)) / 1024:.2f}",
            f"{calc_hei(inf_t, n_test, obj):.2f}",
        ]

    ci_lbl = f"{int(target_cov*100)}% CI"
    rows = {
        "Metric":   [
            f"CCE ({ci_lbl})", f"APIW ({ci_lbl})",
            f"CVaR (tail > predicted P{tail_pct})",
            "Train Time (s)", "Latency (us/sample)", "Model Size (KB)", "HEI",
        ],
        "DDT":      fmt_row(ddt_q05,  ddt_q95,  ddt_inf_t,  ddt,   ddt_fit_t),
        "DDT-Log":  fmt_row(ddt_log_q05, ddt_log_q95, ddt_log_inf_t, ddt_log, ddt_log_fit_t),
        "DDT-Hybrid": fmt_row(ddt_hyb_q05, ddt_hyb_q95, ddt_hyb_inf_t, ddt_hyb, ddt_hyb_fit_t),
        "DDT-Cal":    fmt_row(ddt_cal_q05, ddt_cal_q95, ddt_cal_inf_t, ddt_cal, ddt_cal_fit_t),
        "LightGBM": fmt_row(lgbm_q05, lgbm_q95, lgbm_inf_t, lgbms, lgbm_fit_t),
        "XGBoost":  fmt_row(xgb_q05,  xgb_q95,  xgb_inf_t,  xgbs,  xgb_fit_t),
        "CatBoost": fmt_row(cat_q05,  cat_q95,  cat_inf_t,  cat,   cat_fit_t),
    }
    df = pd.DataFrame(rows).set_index("Metric")
    display(HTML(f"<h4>{profile['name']} ({preproc_name})</h4>"))
    display(df)
    return f"=== {profile['name']} | {preproc_name} ===\n" + df.to_string() + "\n\n"


In [ ]:
all_results_text = ""

# Fixed feature scaling
FEATURE_SCALING = "none"

for preproc in ["standard","winsorized"]: # Add "standard" or "none" for further comparison
    X_tr_s, X_te_s, y_tr_s, y_scaler = preprocess_pipeline(
        X_train_raw, X_test_raw, y_train, 
        target_strategy=preproc, 
        feature_scaling=FEATURE_SCALING
    )
    
    for prof_key, prof in BENCHMARK_PROFILES.items():
        res_text = run_benchmark(prof_key, prof, X_tr_s, X_te_s, y_tr_s, y_test, preproc, y_scaler=y_scaler)
        all_results_text += res_text


c:\Users\mmila\Documents\GitHub\ddt-core-public\.venv\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(



Profile: 1-vs-1 Representational Capacity (The Theoretical Duel) | Preprocessing: standard


,DDT,DDT-Log,DDT-Hybrid,DDT-Cal,LightGBM,XGBoost,CatBoost
Metric,,,,,,,
CCE (89% CI),12.97%,10.87%,9.65%,12.73%,10.69%,13.00%,13.12%
APIW (89% CI),57.73,57.38,56.52,62.32,55.96,55.98,55.97
CVaR (tail > predicted P95),95.21,96.67,97.78,108.20,106.59,106.10,105.96
Train Time (s),0.10,0.11,0.11,0.10,0.14,0.18,0.11
Latency (us/sample),0.13,0.18,0.20,0.57,0.18,0.05,0.07
Model Size (KB),203.01,206.71,209.34,351.98,19.99,36.20,8.09
HEI,25.39,36.84,42.03,201.27,3.62,1.94,0.56



Profile: Edge AI & Micro-Kernel (The Silicon Test) | Preprocessing: standard


,DDT,DDT-Log,DDT-Hybrid,DDT-Cal,LightGBM,XGBoost,CatBoost
Metric,,,,,,,
CCE (89% CI),11.51%,9.19%,9.38%,13.19%,10.69%,13.00%,13.12%
APIW (89% CI),55.23,56.19,53.94,59.58,55.96,55.98,55.97
CVaR (tail > predicted P95),102.52,103.03,102.05,110.04,106.59,106.10,105.96
Train Time (s),0.10,0.08,0.08,0.10,0.17,0.22,0.13
Latency (us/sample),0.10,0.12,0.17,0.33,0.18,0.09,0.10
Model Size (KB),20.91,21.97,21.97,197.84,19.99,36.20,8.09
HEI,2.19,2.56,3.70,65.25,3.70,3.22,0.79


...


Profile: 1-vs-1 Representational Capacity (The Theoretical Duel) | Preprocessing: winsorized


,DDT,DDT-Log,DDT-Hybrid,DDT-Cal,LightGBM,XGBoost,CatBoost
Metric,,,,,,,
CCE (89% CI),11.99%,11.59%,11.20%,14.42%,10.69%,12.98%,13.12%
APIW (89% CI),57.02,57.21,56.51,60.88,55.96,55.98,55.97
CVaR (tail > predicted P95),96.59,96.53,98.71,107.20,106.59,106.10,105.96
Train Time (s),0.13,0.13,0.11,0.11,0.16,0.19,0.11
Latency (us/sample),0.16,0.18,0.21,0.61,0.19,0.07,0.10
Model Size (KB),203.54,204.60,210.40,349.35,20.00,36.21,8.09
HEI,32.94,37.65,45.19,213.35,3.86,2.52,0.79



Profile: Edge AI & Micro-Kernel (The Silicon Test) | Preprocessing: winsorized


,DDT,DDT-Log,DDT-Hybrid,DDT-Cal,LightGBM,XGBoost,CatBoost
Metric,,,,,,,
CCE (89% CI),9.83%,10.69%,9.54%,13.61%,10.69%,12.98%,13.12%
APIW (89% CI),56.10,56.19,55.02,58.86,55.96,55.98,55.97
CVaR (tail > predicted P95),102.54,103.10,102.81,108.79,106.59,106.10,105.96
Train Time (s),0.13,0.10,0.10,0.12,0.27,0.28,0.12
Latency (us/sample),0.15,0.18,0.20,0.70,0.20,0.11,0.10
Model Size (KB),20.91,21.97,21.97,197.84,20.00,36.21,8.09
HEI,3.08,3.91,4.50,138.04,4.09,3.96,0.83



Profile: Production Battle (Single-Tree DDT vs. 100-Tree Boosting) | Preprocessing: winsorized


In [11]:
print("--- FINAL COMPILED RESULTS ---")
print(all_results_text)


--- FINAL COMPILED RESULTS ---
=== 1-vs-1 Representational Capacity (The Theoretical Duel) | standard ===
                                DDT DDT-Log DDT-Hybrid DDT-Cal LightGBM XGBoost CatBoost
Metric                                                                                  
CCE (89% CI)                  9.78%  15.08%     10.38%  19.72%   10.69%  13.00%   13.12%
APIW (89% CI)                 55.79   54.63      57.91   72.55    55.96   55.98    55.97
CVaR (tail > predicted P95)  101.36  101.87     103.06  119.99   106.59  106.10   105.96
Train Time (s)                 0.07    0.06       0.07    0.08     0.16    0.19     0.12
Latency (us/sample)            0.07    0.07       0.09    0.51     0.19    0.06     0.07
Model Size (KB)                7.59    7.90       7.89  183.76    19.99   36.20     8.09
HEI                            0.55    0.58       0.69   93.13     3.78    2.10     0.58

=== Edge AI & Micro-Kernel (The Silicon Test) | standard ===
                               